In [1]:
import pandas as pd
import numpy as np
import re

import nltk
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
df = pd.read_csv("complaints.csv")  # change name if needed

df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2019-06-13,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,CAPITAL ONE FINANCIAL CORPORATION,PA,186XX,NaN,Consent not provided,Web,2019-06-13,Closed with explanation,Yes,NaN,3274605
1,2019-11-01,Vehicle loan or lease,Loan,Struggling to pay your loan,Denied request to lower payments,I contacted Ally on Friday XX/XX/XXXX after fa...,Company has responded to the consumer and the ...,ALLY FINANCIAL INC.,NJ,088XX,NaN,Consent provided,Web,2019-11-01,Closed with explanation,Yes,NaN,3425257
2,2019-04-01,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Account status incorrect,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",PA,19067,NaN,Consent not provided,Web,2019-04-01,Closed with explanation,Yes,NaN,3198225
3,2021-11-01,"Credit reporting, credit repair services, or o...",Credit reporting,Problem with a credit reporting company's inve...,Was not notified of investigation status or re...,NaN,NaN,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",GA,31707,NaN,NaN,Web,2021-11-01,In progress,Yes,NaN,4863965
4,2021-11-02,Debt collection,Medical debt,Took or threatened to take negative or legal a...,Threatened or suggested your credit would be d...,NaN,NaN,"Medical Data Systems, Inc.",VA,22033,NaN,NaN,Web,2021-11-02,In progress,Yes,NaN,4866449


In [3]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [4]:
def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]
    
    return " ".join(words)

In [5]:
df["clean_text"] = df["Consumer complaint narrative"].apply(clean_text)

df[["Consumer complaint narrative", "clean_text"]].head()

,Consumer complaint narrative,clean_text
0,NaN,
1,I contacted Ally on Friday XX/XX/XXXX after fa...,contacted ally friday xxxxxxxx falling behind ...
2,NaN,
3,NaN,
4,NaN,


In [6]:
bow_vectorizer = CountVectorizer(max_features=5000)
X_bow = bow_vectorizer.fit_transform(df["clean_text"])

X_bow.shape

(2326246, 5000)

In [7]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vectorizer.fit_transform(df["clean_text"])

X_tfidf.shape

(2326246, 5000)

In [8]:
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X_bow)

words_bow = bow_vectorizer.get_feature_names_out()

for i, topic in enumerate(lda.components_):
    print(f"\nLDA Topic {i}")
    top_indices = topic.argsort()[-10:][::-1]
    print([words_bow[j] for j in top_indices])


LDA Topic 0
['xxxx', 'payment', 'account', 'would', 'bank', 'xxxxxxxx', 'told', 'card', 'time', 'called']

LDA Topic 1
['xxxx', 'loan', 'mortgage', 'home', 'company', 'payment', 'property', 'document', 'information', 'year']

LDA Topic 2
['credit', 'xxxx', 'debt', 'report', 'collection', 'letter', 'company', 'information', 'reporting', 'sent']

LDA Topic 3
['account', 'information', 'credit', 'report', 'consumer', 'reporting', 'identity', 'theft', 'section', 'file']

LDA Topic 4
['xxxx', 'account', 'credit', 'xxxxxxxx', 'report', 'inquiry', 'date', 'reporting', 'information', 'balance']


In [9]:
nmf = NMF(n_components=5, random_state=42)
nmf.fit(X_tfidf)

words_tfidf = tfidf_vectorizer.get_feature_names_out()

for i, topic in enumerate(nmf.components_):
    print(f"\nNMF Topic {i}")
    top_indices = topic.argsort()[-10:][::-1]
    print([words_tfidf[j] for j in top_indices])


NMF Topic 0
['xxxx', 'xxxxxxxx', 'number', 'address', 'name', 'date', 'inquiry', 'following', 'xxxxxxxxxxxx', 'experian']

NMF Topic 1
['credit', 'report', 'information', 'reporting', 'bureau', 'inquiry', 'item', 'dispute', 'inaccurate', 'equifax']

NMF Topic 2
['payment', 'loan', 'xxxxxxxx', 'would', 'told', 'late', 'time', 'month', 'bank', 'mortgage']

NMF Topic 3
['account', 'bank', 'opened', 'closed', 'balance', 'open', 'fraudulent', 'please', 'charge', 'card']

NMF Topic 4
['debt', 'collection', 'company', 'letter', 'agency', 'validation', 'sent', 'owe', 'collect', 'received']


In [10]:
print("BoW shape:", X_bow.shape)
print("TF-IDF shape:", X_tfidf.shape)

BoW shape: (2326246, 5000)
TF-IDF shape: (2326246, 5000)
